<a href="https://colab.research.google.com/github/GunaPalanivel/Praxis/blob/main/praxis_grpo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praxis GRPO Colab Training Notebook

This notebook runs a lightweight trajectory-reward loop against Praxis over HTTP: a softmax policy over a fixed `ACTION_POOL`, with group-relative advantages (GRPO-style). It is **not** `GRPOTrainer` fine-tuning a loaded language model; a later cell initializes TRL's `GRPOConfig` only for API parity with local `train_praxis_grpo.py`.

Outputs saved in Colab working directory:
- `reward_curve.png`
- `loss_curve.png`


In [1]:
!pip install unsloth openenv trl matplotlib numpy requests transformers accelerate peft bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [2]:
import math
import os
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import requests
import trl

PRAXIS_BASE_URL = "https://gp5901-praxis.hf.space"
TASK_NAME = "single-service-alert"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.environ.get("HF_TOKEN", "")
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("trl version:", getattr(trl, "__version__", "unknown"))
print("model:", MODEL_ID)
print("hf token loaded:", bool(HF_TOKEN))

trl version: 0.24.0
model: Qwen/Qwen2.5-7B-Instruct
hf token loaded: False


In [3]:
def api_get(path: str):
    resp = requests.get(f"{PRAXIS_BASE_URL}{path}", timeout=30)
    resp.raise_for_status()
    return resp.json()


def api_post(path: str, payload: dict, headers: dict | None = None):
    resp = requests.post(f"{PRAXIS_BASE_URL}{path}", json=payload, headers=headers or {}, timeout=30)
    resp.raise_for_status()
    return resp.json()


health = api_get("/health")
tasks = api_get("/tasks")
reset = api_post("/reset", {"task_name": TASK_NAME})
session_id = reset.get("session_id", "")
headers = {"x-session-id": session_id} if session_id else {}
step = api_post("/step", {"command": "query_logs service=auth timerange=5m"}, headers=headers)

print("health:", health)
print("tasks:", tasks)
print("step reward:", step["reward"], "done:", step["done"])

health: {'status': 'healthy', 'environment': 'praxis-env', 'version': '1.0.0', 'available_tasks': ['ambiguous-incident', 'cascading-failure', 'cascading-platform-failure', 'memory-leak', 'procedural-incident', 'single-service-alert']}
tasks: {'tasks': ['ambiguous-incident', 'cascading-failure', 'cascading-platform-failure', 'memory-leak', 'procedural-incident', 'single-service-alert']}
step reward: 0.08 done: False


In [4]:
ACTION_POOL = [
    "query_logs service=auth timerange=5m",
    "check_metrics service=auth metric=error_rate",
    "check_config service=auth",
    "diagnose root_cause=bad_config",
    "restart_service service=auth",
    "rollback_deploy service=auth",
    "escalate reason=need_senior_support",
]

ACTION_TO_IDX = {a: i for i, a in enumerate(ACTION_POOL)}


def softmax(logits: np.ndarray) -> np.ndarray:
    x = logits - np.max(logits)
    e = np.exp(x)
    return e / np.sum(e)


def run_episode_with_policy(logits: np.ndarray, max_steps: int = 8):
    reset_payload = api_post("/reset", {"task_name": TASK_NAME})
    sid = reset_payload.get("session_id", "")
    headers = {"x-session-id": sid} if sid else {}

    rewards = []
    chosen_idxs = []
    done = False

    for _ in range(max_steps):
        probs = softmax(logits)
        idx = int(np.random.choice(len(ACTION_POOL), p=probs))
        command = ACTION_POOL[idx]
        step_payload = api_post("/step", {"command": command}, headers=headers)
        reward = float(step_payload["reward"])
        done = bool(step_payload["done"])

        rewards.append(reward)
        chosen_idxs.append(idx)

        if done:
            break

    return rewards, chosen_idxs


In [5]:
def evaluate_policy(logits: np.ndarray, episodes: int = 8, max_steps: int = 8):
    episode_means = []
    for _ in range(episodes):
        rewards, _ = run_episode_with_policy(logits, max_steps=max_steps)
        if rewards:
            episode_means.append(float(np.mean(rewards)))
    return episode_means


baseline_logits = np.zeros(len(ACTION_POOL), dtype=np.float64)
baseline_rewards = evaluate_policy(baseline_logits, episodes=12, max_steps=8)
print("baseline mean reward:", float(np.mean(baseline_rewards)))

baseline mean reward: 0.10700892857142857


In [6]:
try:
    from trl import GRPOConfig

    trl_grpo_config = GRPOConfig(
        learning_rate=1e-5,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        max_prompt_length=512,
        max_completion_length=128,
    )
    print("TRL GRPOConfig initialized")
except Exception as exc:
    trl_grpo_config = None
    print("TRL GRPOConfig unavailable in this runtime:", exc)


@dataclass
class TrainConfig:
    episodes: int = 40
    group_size: int = 4
    max_steps: int = 8
    lr: float = 0.08


cfg = TrainConfig()
policy_logits = np.zeros(len(ACTION_POOL), dtype=np.float64)

reward_curve = []
loss_curve = []

for episode_idx in range(cfg.episodes):
    group_returns = []
    group_actions = []

    for _ in range(cfg.group_size):
        rewards, chosen_idxs = run_episode_with_policy(policy_logits, max_steps=cfg.max_steps)
        ep_return = float(np.mean(rewards)) if rewards else 0.01
        group_returns.append(ep_return)
        group_actions.append(chosen_idxs)

    returns_arr = np.array(group_returns, dtype=np.float64)
    baseline = float(np.mean(returns_arr))
    std = float(np.std(returns_arr) + 1e-8)
    advantages = (returns_arr - baseline) / std

    probs = softmax(policy_logits)
    grad = np.zeros_like(policy_logits)
    loss = 0.0

    for adv, actions in zip(advantages, group_actions):
        if not actions:
            continue
        for act in actions:
            one_hot = np.zeros_like(policy_logits)
            one_hot[act] = 1.0
            grad += adv * (one_hot - probs)
            loss += -adv * math.log(max(probs[act], 1e-8))

    grad /= max(1, sum(len(a) for a in group_actions))
    loss /= max(1, sum(len(a) for a in group_actions))

    policy_logits += cfg.lr * grad

    reward_curve.append(float(np.mean(group_returns)))
    loss_curve.append(float(loss))

print("training complete")
print("last train reward:", reward_curve[-1])
print("last train loss:", loss_curve[-1])

TRL GRPOConfig unavailable in this runtime: Your setup doesn't support bf16/gpu. You need to assign use_cpu if you want to train the model on CPU.


HTTPError: 429 Client Error: Too Many Requests for url: https://gp5901-praxis.hf.space/step

In [ ]:
trained_rewards = evaluate_policy(policy_logits, episodes=12, max_steps=8)
print("trained mean reward:", float(np.mean(trained_rewards)))

x_baseline = np.arange(1, len(baseline_rewards) + 1)
x_trained = np.arange(1, len(trained_rewards) + 1)

plt.figure(figsize=(10, 5))
plt.plot(x_baseline, baseline_rewards, color="red", label="Untrained Baseline")
plt.plot(x_trained, trained_rewards, color="green", label="Trained Agent")
plt.xlabel("Training Step / Episode")
plt.ylabel("Mean Episode Reward")
plt.title("Praxis Reward Comparison: Baseline vs Trained")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("reward_curve.png", dpi=150)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(np.arange(1, len(loss_curve) + 1), loss_curve, color="blue", label="Training Loss")
plt.xlabel("Training Step / Episode")
plt.ylabel("Loss")
plt.title("GRPO Training Loss Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

print("Saved files: reward_curve.png, loss_curve.png")

In [ ]:
from google.colab import files
files.download("reward_curve.png")
files.download("loss_curve.png")